[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/06-dsa-interview/01_arrays.ipynb)

*DSA Interview Handbook · Notebook 1 of 14*

# Arrays & Hashing

Indexing, prefix sums, hash maps and in-place tricks — the base of every other topic.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Hash maps trade a little memory for a huge speed-up: remember what you've seen, rather than searching again.

**What interviewers really test here:** Spotting hash-map and prefix-sum patterns, and in-place tricks.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. What an array really is *(Basics)*
- [ ] 2. Insert, delete and search — what they cost *(Basics)*
- [ ] 3. Hash maps and hash sets *(Core)*
- [ ] 4. Pattern: remember what you have seen *(Core)*
- [ ] 5. Prefix sums *(Core)*
- [ ] 6. Running state: Kadane and one-pass scans *(Advanced)*
- [ ] 7. In-place tricks and matrices *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

An **array** is a **linear data structure** that stores a collection of elements in **contiguous** (side-by-side) memory locations. Each element is identified by an **index**; in most languages, including Python, indexing starts at 0.



Because the elements are contiguous and of equal size, the address of element i is computed directly:



```text
address(a[i]) = base_address + i × element_size
```



This formula is why arrays support **random access** — reaching any index in O(1) time.



A **hash table** is a data structure that maps **keys** to **values** using a **hash function** to compute an index (bucket) where each key is stored. Python's `dict` (key → value) and `set` (keys only) are hash tables.

### Types of arrays

| Type | Meaning | Example |
| --- | --- | --- |
| **Static array** | size fixed when created | C `int a[10]`, Java `int[]` |
| **Dynamic array** | grows automatically by reallocating a larger block | Python `list`, Java `ArrayList`, C++ `vector` |
| **1-D array** | a single row of elements | `[3, 1, 4]` |
| **2-D array (matrix)** | rows × columns; stored row by row (row-major) in C/Python | `grid[r][c]` |
| **Multi-dimensional** | 3 or more indices | images (height × width × colour) |
| **Jagged array** | rows of different lengths | `[[1], [2, 3], [4, 5, 6]]` |



**Strings** are arrays of characters. In Python they are **immutable**: `s[0] = "x"` is an error, and `s += c` in a loop copies the string each time (O(n²) overall) — build a list and `"".join()` it instead.

### Operations and their time complexity

| Operation | How it works | Time |
| --- | --- | --- |
| Access `a[i]` | address formula | O(1) |
| Update `a[i] = x` | address formula | O(1) |
| Traverse | visit each element once | O(n) |
| Linear search | compare with each element | O(n) |
| Binary search (sorted) | halve the range each step | O(log n) |
| Insert at end (dynamic) | use spare capacity; resize when full | O(1) amortised |
| Insert at index i | shift n − i elements right | O(n) |
| Delete at index i | shift n − i − 1 elements left | O(n) |
| Sort | Timsort in Python | O(n log n) |



#### Amortised O(1) append — why



When a dynamic array is full it allocates double the capacity and copies every element. Starting from capacity 1 and appending n items, the copies cost 1 + 2 + 4 + … + n/2 &lt; n in total. Adding the n writes themselves, n appends cost less than 2n operations, so each append costs O(1) **on average**, even though one particular append can cost O(n).

### Hashing in depth

#### Hash function



Converts a key into an integer. A good hash function is **deterministic** (same key → same hash), **fast**, and spreads keys **uniformly** across buckets. The bucket is usually `hash(key) % number_of_buckets`.



#### Collisions and how they are resolved



A **collision** happens when two keys map to the same bucket. Two main strategies:




- **Separate chaining**: each bucket stores a list of entries. Lookup scans that short list.
- **Open addressing**: all entries live in the array itself; on collision, **probe** other slots — linear probing (next slot), quadratic probing, or double hashing. Python's `dict` uses open addressing.



#### Load factor and rehashing



**Load factor** α = number of entries ÷ number of buckets. As α grows, collisions increase. When α passes a threshold (about 2/3 in Python), the table **rehashes**: allocates more buckets and reinserts every key. Like array resizing, this keeps operations O(1) amortised.



| Operation | Average | Worst case (all keys collide) |
| --- | --- | --- |
| insert / lookup / delete | O(1) | O(n) |



Only **immutable** (hashable) objects can be keys: int, str, tuple, frozenset. Lists and dicts can't — their hash would change if they changed.

### Advantages and disadvantages

**👍 Pros**

##### Arrays — advantages

- O(1) random access by index
- Cache-friendly: neighbouring elements are loaded together, so traversal is very fast in practice
- No extra memory per element (no pointers)
- Simple to use; the basis of most other structures



**👎 Cons**

##### Arrays — disadvantages

- Insert/delete in the middle costs O(n)
- Static arrays have a fixed size; dynamic arrays waste some spare capacity
- Searching an unsorted array is O(n)
- Need one large contiguous block of memory



**👍 Pros**

##### Hash tables — advantages

- O(1) average insert, lookup and delete by key
- Keys can be strings, tuples, etc., not just 0..n−1
- Ideal for counting, de-duplication and caching



**👎 Cons**

##### Hash tables — disadvantages

- No ordering by key (no "smallest key" or range query)
- O(n) worst case with bad hashing
- More memory than an array (empty buckets, stored hashes)
- Keys must be hashable

### Applications

- **Arrays**: lookup tables, images and matrices, buffers, implementing stacks, queues, heaps and hash tables themselves, dynamic programming tables.
- **Prefix sums**: fast range-sum queries in analytics, 2-D prefix sums for image processing (summed-area tables).
- **Hash maps**: database indexes, caches (memoisation, LRU caches), counting word frequencies, symbol tables in compilers, de-duplicating records, sets of visited states in search.

### Array vs linked list vs hash table

|   | Array | Linked list | Hash table |
| --- | --- | --- | --- |
| Access by position | O(1) | O(n) | — |
| Search by value / key | O(n), O(log n) if sorted | O(n) | O(1) avg |
| Insert / delete at front | O(n) | O(1) | O(1) avg |
| Insert / delete at end | O(1) amortised | O(1) with tail pointer | — |
| Keeps order | yes | yes | insertion order in Python 3.7+, not sorted |
| Memory | compact | extra pointer per node | extra empty buckets |

---
# Part 2: Lessons, step by step

## Lesson 1: What an array really is

*Level: Basics*

An **array** is a row of boxes placed side by side in memory. Every box has a number called its **index**, starting at 0. In Python, a `list` is an array.



```text
nums  = [ 7 | 3 | 9 | 4 ]
index    0   1   2   3
```



Because the boxes are next to each other and all the same size, the computer can jump straight to box `i` with one calculation: *start address + i × box size*. That is why `nums[i]` takes the same time for index 2 or index 2,000,000. We write this as **O(1)** — "constant time".



#### How a Python list grows



A list keeps some spare boxes at the end. `append` just fills the next spare box (fast). When the spare boxes run out, Python makes a **bigger** block (about double), copies everything, and continues. Copying is slow, but it happens rarely, so on average `append` is still O(1). This is called **amortised O(1)**.



**💡 Tip:** Rule of thumb: reading or writing by index and appending at the end are cheap. Anything that has to move many elements is expensive.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def append_growth():
    capacity, size, boxes = 1, 0, [None]
    for x in [5, 8, 2, 6, 1]:
        if size == capacity:
            capacity *= 2
            boxes = boxes[:size] + [None] * (capacity - size)
        boxes[size] = x
        size += 1
    return boxes


append_growth()

[5, 8, 2, 6, 1, None, None, None]

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. append(5) fills box 0
2. No space left → allocate 2 boxes and copy the 1 old values
3. append(8) fills box 1
4. No space left → allocate 4 boxes and copy the 2 old values
5. append(2) fills box 2
6. append(6) fills box 3
7. No space left → allocate 8 boxes and copy the 4 old values
8. append(1) fills box 4

</details>


**✏️ Your turn:** you now have what you need for [Q2](#q2). Try one now, or carry on and do them in Part 3.

## Lesson 2: Insert, delete and search — what they cost

*Level: Basics*

Adding or removing at the **end** is cheap. Doing it at the **front or middle** is expensive, because every element after that spot must move one box over.



| Operation | Python | Cost | Why |
| --- | --- | --- | --- |
| read / write index i | `a[i]` | O(1) | direct jump |
| add at end | `a.append(x)` | O(1)\* | fills a spare box |
| remove at end | `a.pop()` | O(1) | nothing moves |
| insert at i | `a.insert(i, x)` | O(n) | shift everything after i right |
| remove at i | `a.pop(i)` | O(n) | shift everything after i left |
| search for a value | `x in a`, `a.index(x)` | O(n) | must look at every box |
| copy a slice | `a[i:j]` | O(j − i) | creates a new list |



**⚠️ Watch out:** A hidden `x in a` or `a.pop(0)` inside a loop turns an O(n) solution into O(n²). Interviewers look for this.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def insert_shift():
    a = [10, 20, 30, 40, 50]
    i, x = 1, 15
    a.append(None)
    for k in range(len(a) - 1, i, -1):
        a[k] = a[k - 1]
    a[i] = x
    return a


insert_shift()

[10, 15, 20, 30, 40, 50]

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Shift a[4] one box right
2. Shift a[3] one box right
3. Shift a[2] one box right
4. Shift a[1] one box right
5. Now box 1 is free → place 15. That took 4 moves.

</details>


## Lesson 3: Hash maps and hash sets

*Level: Core*

Searching a list for a value is O(n). A **hash set** (`set`) and a **hash map** (`dict`) make that search O(1) on average.



#### How it works




1. A **hash function** turns the key into a number, e.g. `hash(42)`.
2. That number, modulo the number of buckets, picks a **bucket**.
3. The key (and its value) is stored in that bucket. To look it up later, compute the same bucket and check only there.



Two keys can land in the same bucket — a **collision**. The bucket then holds a short list. Python grows the table when it gets too full, so buckets stay short and lookups stay O(1) on average (O(n) only in a very unlucky worst case).



| You want | Use |
| --- | --- |
| "have I seen x?" | `seen = set()`, `x in seen` |
| remember something about x (index, count) | `d = {}`, `d[x] = i` |
| count things | `collections.Counter(nums)` |
| group things | `collections.defaultdict(list)` |



**💡 Tip:** Keys must be **hashable** (unchangeable): numbers, strings, tuples. A list can't be a key — convert it with `tuple(lst)`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def hashing_demo():
    buckets = [[] for _ in range(5)]
    for key in [12, 7, 33, 22, 5, 18]:
        b = key % 5
        buckets[b].append(key)
    key = 22
    b = key % 5
    return buckets


hashing_demo()

[[5], [], [12, 7, 22], [33, 18], []]

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. hash(12) % 5 = 2 → put 12 in bucket 2
2. hash(7) % 5 = 2 → put 7 in bucket 2
3. hash(33) % 5 = 3 → put 33 in bucket 3
4. hash(22) % 5 = 2 → put 22 in bucket 2
5. hash(5) % 5 = 0 → put 5 in bucket 0
6. hash(18) % 5 = 3 → put 18 in bucket 3
7. Look up 22: go straight to bucket 2 and check only [12, 7, 22]

</details>


**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q3](#q3). Try one now, or carry on and do them in Part 3.

## Lesson 4: Pattern: remember what you have seen

*Level: Core*

Many "find a pair" problems look like they need two loops. The trick: walk the array **once** and keep a dict of what you've already passed. For each new element, ask the dict a question in O(1).



```python
# Two Sum: is there an earlier number equal to target - x?
seen = {}
for i, x in enumerate(nums):
    if target - x in seen:
        return [seen[target - x], i]
    seen[x] = i
```



The same idea covers many problems — only the *question* changes:




- **Duplicates**: "is x already in the set?"
- **Anagrams**: count letters with a dict; two words are anagrams when their counts match.
- **Grouping**: build a key that is equal for items in the same group (sorted letters, or a 26-count tuple), and use `defaultdict(list)`.
- **Top-k frequent**: count first, then pick the largest counts (bucket sort makes it O(n)).

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q8](#q8), [Q9](#q9), [Q11](#q11). Try one now, or carry on and do them in Part 3.

## Lesson 5: Prefix sums

*Level: Core*

A **prefix sum** array stores running totals: `pre[i]` = sum of the first `i` elements. Building it is one pass, O(n).



```text
nums = [3, 1, 4, 1, 5]
pre  = [0, 3, 4, 8, 9, 14]      pre[0] = 0 (nothing added yet)
```



Then the sum of any range `nums[l..r]` is just **one subtraction**: `pre[r + 1] − pre[l]`. Without it, each range sum is a loop.



#### Prefix sums + hash map



To count subarrays that sum to `k`: a subarray ending at `i` sums to `k` when some earlier prefix equals `pre − k`. Keep a dict counting each prefix value seen so far (start with `{0: 1}` for the empty prefix). This works even with negative numbers, where sliding window fails.



The same "running" idea gives **prefix products** (product of array except self) and **suffix** arrays scanned from the right.

**✏️ Your turn:** you now have what you need for [Q10](#q10), [Q14](#q14). Try one now, or carry on and do them in Part 3.

## Lesson 6: Running state: Kadane and one-pass scans

*Level: Advanced*

Some problems only need a few variables updated as you walk the array — no extra arrays at all.



#### Kadane's algorithm (maximum subarray)



Keep `cur` = the best sum of a subarray that **ends at the current index**. At each element you have two choices: extend the previous subarray or start fresh here.



```python
cur = best = nums[0]
for x in nums[1:]:
    cur = max(x, cur + x)   # extend or restart
    best = max(best, cur)
```



Why restart? If `cur` is negative, carrying it only makes the next sum smaller.



#### Other one-pass ideas




- **Running minimum** (best time to buy and sell): track the cheapest price so far; profit today = price − cheapest.
- **Voting** (majority element): a candidate and a counter; equal → +1, different → −1.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def kadane_lesson():
    nums = [-2, 1, -3, 4, -1, 2, 1, -5]
    cur = best = nums[0]
    for i in range(1, len(nums)):
        x = nums[i]
        choice = "restart at " + str(x) if x > cur + x else "extend to " + str(cur + x)
        cur = max(x, cur + x)
        best = max(best, cur)
    return best


kadane_lesson()

6

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. x = 1: restart at 1
2. x = -3: extend to -2
3. x = 4: restart at 4
4. x = -1: extend to 3
5. x = 2: extend to 5
6. x = 1: extend to 6
7. x = -5: extend to 1

</details>


**✏️ Your turn:** you now have what you need for [Q12](#q12), [Q4](#q4), [Q5](#q5). Try one now, or carry on and do them in Part 3.

## Lesson 7: In-place tricks and matrices

*Level: Advanced*

"Use O(1) extra space" means: no new arrays — rearrange the input itself.




- **Write pointer**: one index reads, another writes the elements you keep (move zeroes).
- **Reverse trick**: rotating by k = reverse all, reverse first k, reverse the rest.
- **Value as index**: when values are 1..n, value v can point to index v (find duplicate as a cycle, or mark seen by making `nums[v−1]` negative).
- **Three-way partition** (Dutch flag): pointers for the 0-zone, the 2-zone and the scanner.



#### 2-D arrays (matrices)



A matrix is a list of rows: `grid[r][c]`. `m = len(grid)` rows, `n = len(grid[0])` columns. Common moves:




- **Transpose**: swap `grid[r][c]` with `grid[c][r]` for `c > r`.
- **Rotate 90° clockwise** = transpose + reverse each row.
- **Spiral**: keep four walls (top, bottom, left, right) and shrink them.
- **Flatten an index**: cell k ↔ `(k // n, k % n)`.



**⚠️ Watch out:** `[[0] * n] * m` creates m references to the **same** row. Use `[[0] * n for _ in range(m)]`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [6]:
def mark_negative():
    nums = [4, 3, 2, 7, 8, 2, 3, 1]
    dup = []
    for i in range(len(nums)):
        v = abs(nums[i])
        if nums[v - 1] < 0:
            dup.append(v)
        else:
            nums[v - 1] = -nums[v - 1]
    return dup


mark_negative()

[2, 3]

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Value 4: mark box 3 as seen by making it negative
2. Value 3: mark box 2 as seen by making it negative
3. Value 2: mark box 1 as seen by making it negative
4. Value 7: mark box 6 as seen by making it negative
5. Value 8: mark box 7 as seen by making it negative
6. Value 2: box 1 is already negative → 2 is a duplicate
7. Value 3: box 2 is already negative → 3 is a duplicate
8. Value 1: mark box 0 as seen by making it negative

</details>


**✏️ Your turn:** you now have what you need for [Q6](#q6), [Q13](#q13), [Q15](#q15), [Q17](#q17), [Q20](#q20), [Q19](#q19), [Q18](#q18). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Two Sum

🟢 Warm-up · *Hash map*

Given an array `nums` and an integer `target`, return the indices of the two numbers that add up to `target`. Exactly one answer exists, and you may not use the same element twice.

**Examples**

- `nums = [2, 7, 11, 15], target = 9` → `[0, 1]`
- `nums = [3, 2, 4], target = 6` → `[1, 2]`
- `nums = [3, 3], target = 6` → `[0, 1]`

In [ ]:
def two_sum(nums, target):
    """
    Given an array nums and an integer target, return the indices of the two
    numbers that add up to target. Exactly one answer exists, and you may not
    use the same element twice.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(two_sum, [(([2, 7, 11, 15], 9), [0, 1]), (([3, 2, 4], 6), [1, 2]), (([3, 3], 6), [0, 1])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every pair (i, j) with i &lt; j and return the first pair whose sum is target.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Scan once and keep a map value → index. For each x the partner we need is target − x; if it is already in the map, the earlier index and the current index are the answer.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every pair (i, j) with i &lt; j and return the first pair whose sum is target.

```python
def two_sum(nums, target):
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] + nums[j] == target:
                return [i, j]
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Scan once and keep a map value → index. For each x the partner we need is target − x; if it is already in the map, the earlier index and the current index are the answer.

```python
def two_sum(nums, target):
    seen = {}  # value -> index
    for i, x in enumerate(nums):
        need = target - x
        if need in seen:
            return [seen[need], i]
        seen[x] = i
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every pair (i, j) with i &lt; j and return the first pair whose sum is target. That costs O(n²) time and O(1) space."

"I can do better: Scan once and keep a map value → index. For each x the partner we need is target − x; if it is already in the map, the earlier index and the current index are the answer. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [3, 8, 5, 1, 6], target = 7`

1. x = 3. Need 7 − 3 = 4. Is 4 in seen?
2. x = 8. Need 7 − 8 = -1. Is -1 in seen?
3. x = 5. Need 7 − 5 = 2. Is 2 in seen?
4. x = 1. Need 7 − 1 = 6. Is 6 in seen?
5. x = 6. Need 7 − 6 = 1. Is 1 in seen?
6. Yes — 1 is at index 3. Answer [3, 4]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/two-sum)

<a id="q2"></a>
### Q2. Contains Duplicate

🟢 Warm-up · *Hash set*

Return `True` if any value appears at least twice in `nums`, otherwise `False`.

**Examples**

- `nums = [1, 2, 3, 1]` → `True`
- `nums = [1, 2, 3, 4]` → `False`
- `nums = [1, 1, 1, 3, 3, 4, 3, 2, 4, 2]` → `True`

In [ ]:
def contains_duplicate(nums):
    """
    Return True if any value appears at least twice in nums, otherwise False.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(contains_duplicate, [
    (([1, 2, 3, 1],), True),
    (([1, 2, 3, 4],), False),
    (([1, 1, 1, 3, 3, 4, 3, 2, 4, 2],), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare every pair of elements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Walk the array with a set of values seen so far. The first value already in the set is a duplicate.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Compare every pair of elements.

```python
def contains_duplicate(nums):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] == nums[j]:
                return True
    return False
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Walk the array with a set of values seen so far. The first value already in the set is a duplicate.

```python
def contains_duplicate(nums):
    seen = set()
    for i, x in enumerate(nums):
        if x in seen:
            return True
        seen.add(x)
    return False
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare every pair of elements. That costs O(n²) time and O(1) space."

"I can do better: Walk the array with a set of values seen so far. The first value already in the set is a duplicate. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 2, 3, 1]`

1. Is 1 already in the set?
2. Is 2 already in the set?
3. Is 3 already in the set?
4. Is 1 already in the set?
5. 1 was seen before → duplicate

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/contains-duplicate)

<a id="q3"></a>
### Q3. Valid Anagram

🟢 Warm-up · *Counting*

Return `True` if string `t` is an anagram of `s` — the same letters with the same counts, in any order.

**Examples**

- `s = 'anagram', t = 'nagaram'` → `True`
- `s = 'rat', t = 'car'` → `False`
- `s = 'ab', t = 'a'` → `False`

In [ ]:
def is_anagram(s, t):
    """
    Return True if string t is an anagram of s — the same letters with the same
    counts, in any order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_anagram, [
    (('anagram', 'nagaram'), True),
    (('rat', 'car'), False),
    (('ab', 'a'), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort both strings and compare them.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Count letters: +1 for each letter of s, −1 for each letter of t. They are anagrams exactly when every count ends at 0.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Sort both strings and compare them.

```python
def is_anagram(s, t):
    return sorted(s) == sorted(t)
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Count letters: +1 for each letter of s, −1 for each letter of t. They are anagrams exactly when every count ends at 0.

```python
def is_anagram(s, t):
    if len(s) != len(t):
        return False
    count = {}
    for i in range(len(s)):
        count[s[i]] = count.get(s[i], 0) + 1
        count[t[i]] = count.get(t[i], 0) - 1
    return all(c == 0 for c in count.values())
```

**Cost:** O(n) time, O(1) — 26 letters space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort both strings and compare them. That costs O(n log n) time and O(n) space."

"I can do better: Count letters: +1 for each letter of s, −1 for each letter of t. They are anagrams exactly when every count ends at 0. That's O(n) time and O(1) — 26 letters space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'cat', t = 'act'`

1. +1 for s[0]='c', −1 for t[0]='a'
2. +1 for s[1]='a', −1 for t[1]='c'
3. +1 for s[2]='t', −1 for t[2]='t'
4. All counts are zero?

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/valid-anagram)

<a id="q4"></a>
### Q4. Best Time to Buy and Sell Stock

🟢 Warm-up · *Running minimum*

`prices[i]` is a stock's price on day i. Buy once and sell once later; return the largest profit, or 0 if no profit is possible.

**Examples**

- `prices = [7, 1, 5, 3, 6, 4]` → `5`
- `prices = [7, 6, 4, 3, 1]` → `0`

In [ ]:
def max_profit(prices):
    """
    prices[i] is a stock's price on day i. Buy once and sell once later; return
    the largest profit, or 0 if no profit is possible.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_profit, [
    (([7, 1, 5, 3, 6, 4],), 5),
    (([7, 6, 4, 3, 1],), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every buy day with every later sell day.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Walk the days keeping the cheapest price so far. Selling today earns price − cheapest; keep the best.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every buy day with every later sell day.

```python
def max_profit(prices):
    best = 0
    for i in range(len(prices)):
        for j in range(i + 1, len(prices)):
            best = max(best, prices[j] - prices[i])
    return best
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Walk the days keeping the cheapest price so far. Selling today earns price − cheapest; keep the best.

```python
def max_profit(prices):
    lo, best, lo_i = prices[0], 0, 0
    for i, p in enumerate(prices):
        if p < lo:
            lo, lo_i = p, i
        best = max(best, p - lo)
    return best
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every buy day with every later sell day. That costs O(n²) time and O(1) space."

"I can do better: Walk the days keeping the cheapest price so far. Selling today earns price − cheapest; keep the best. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `prices = [7, 1, 5, 3, 6, 4]`

1. Day 0: cheapest so far 7, selling today earns 0
2. Day 1: cheapest so far 1, selling today earns 0
3. Day 2: cheapest so far 1, selling today earns 4
4. Day 3: cheapest so far 1, selling today earns 2
5. Day 4: cheapest so far 1, selling today earns 5
6. Day 5: cheapest so far 1, selling today earns 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/best-time-to-buy-and-sell-stock)

<a id="q5"></a>
### Q5. Majority Element

🟢 Warm-up · *Boyer–Moore voting*

Return the element that appears more than ⌊n / 2⌋ times. It always exists.

**Examples**

- `nums = [3, 2, 3]` → `3`
- `nums = [2, 2, 1, 1, 1, 2, 2]` → `2`

In [ ]:
def majority_element(nums):
    """
    Return the element that appears more than ⌊n / 2⌋ times. It always exists.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(majority_element, [
    (([3, 2, 3],), 3),
    (([2, 2, 1, 1, 1, 2, 2],), 2),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each element, count how often it appears.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Boyer–Moore: keep a candidate and a counter. Same value → +1, different → −1, counter at 0 → new candidate. The majority survives because it outnumbers everything else combined.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each element, count how often it appears.

```python
def majority_element(nums):
    for x in nums:
        if sum(1 for y in nums if y == x) > len(nums) // 2:
            return x
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Boyer–Moore: keep a candidate and a counter. Same value → +1, different → −1, counter at 0 → new candidate. The majority survives because it outnumbers everything else combined.

```python
def majority_element(nums):
    cand, cnt = None, 0
    for i, x in enumerate(nums):
        if cnt == 0:
            cand = x
        cnt += 1 if x == cand else -1
    return cand
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each element, count how often it appears. That costs O(n²) time and O(1) space."

"I can do better: Boyer–Moore: keep a candidate and a counter. Same value → +1, different → −1, counter at 0 → new candidate. The majority survives because it outnumbers everything else combined. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [2, 2, 1, 1, 1, 2, 2]`

1. x = 2 → candidate 2, count 1
2. x = 2 → candidate 2, count 2
3. x = 1 → candidate 2, count 1
4. x = 1 → candidate 2, count 0
5. x = 1 → candidate 1, count 1
6. x = 2 → candidate 1, count 0
7. x = 2 → candidate 2, count 1

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/majority-element)

<a id="q6"></a>
### Q6. Move Zeroes

🟢 Warm-up · *Write pointer*

Move all 0s in `nums` to the end while keeping the order of the other elements. Do it in place.

**Examples**

- `nums = [0, 1, 0, 3, 12]` → `[1, 3, 12, 0, 0]`
- `nums = [0]` → `[0]`
- `nums = [1, 2]` → `[1, 2]`

In [ ]:
def move_zeroes(nums):
    """
    Move all 0s in nums to the end while keeping the order of the other
    elements. Do it in place.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(move_zeroes, [(([0, 1, 0, 3, 12],), [1, 3, 12, 0, 0]), (([0],), [0]), (([1, 2],), [1, 2])], inplace=0)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build a new list of the non-zeros, pad with zeros, copy it back.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep a write pointer w for the next non-zero slot. When nums[r] is non-zero, swap it into position w and advance w.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Build a new list of the non-zeros, pad with zeros, copy it back.

```python
def move_zeroes(nums):
    rest = [x for x in nums if x != 0]
    rest += [0] * (len(nums) - len(rest))
    nums[:] = rest
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Keep a write pointer w for the next non-zero slot. When nums[r] is non-zero, swap it into position w and advance w.

```python
def move_zeroes(nums):
    w = 0
    for r in range(len(nums)):
        if nums[r] != 0:
            nums[w], nums[r] = nums[r], nums[w]
            w += 1
        else:
            pass
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build a new list of the non-zeros, pad with zeros, copy it back. That costs O(n) time and O(n) space."

"I can do better: Keep a write pointer w for the next non-zero slot. When nums[r] is non-zero, swap it into position w and advance w. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [0, 1, 0, 3, 12]`

1. nums[0] is 0 → skip
2. nums[1] is non-zero → swap into slot w=0
3. nums[2] is 0 → skip
4. nums[3] is non-zero → swap into slot w=1
5. nums[4] is non-zero → swap into slot w=2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/move-zeroes)

<a id="q7"></a>
### Q7. Missing Number

🟢 Warm-up · *Math / XOR*

`nums` holds n distinct numbers from 0..n. Return the one that is missing.

**Examples**

- `nums = [3, 0, 1]` → `2`
- `nums = [0, 1]` → `2`
- `nums = [9, 6, 4, 2, 3, 5, 7, 0, 1]` → `8`

In [ ]:
def missing_number(nums):
    """
    nums holds n distinct numbers from 0..n. Return the one that is missing.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(missing_number, [
    (([3, 0, 1],), 2),
    (([0, 1],), 2),
    (([9, 6, 4, 2, 3, 5, 7, 0, 1],), 8),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each candidate 0..n, search the list for it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The sum of 0..n is n(n+1)/2. Subtract the actual sum; what's left is the missing number.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each candidate 0..n, search the list for it.

```python
def missing_number(nums):
    for v in range(len(nums) + 1):
        if v not in nums:
            return v
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** The sum of 0..n is n(n+1)/2. Subtract the actual sum; what's left is the missing number.

```python
def missing_number(nums):
    n = len(nums)
    expected = n * (n + 1) // 2
    actual = sum(nums)
    return expected - actual
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each candidate 0..n, search the list for it. That costs O(n²) time and O(1) space."

"I can do better: The sum of 0..n is n(n+1)/2. Subtract the actual sum; what's left is the missing number. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/missing-number)

<a id="q8"></a>
### Q8. Group Anagrams

🟡 Interview standard · *Hash map · key design*

Group the strings in `strs` so that anagrams are together. The groups may be returned in any order.

**Examples**

- `strs = ['eat', 'tea', 'tan', 'ate', 'nat', 'bat']` → `[['bat'], ['nat', 'tan'], ['ate', 'eat', 'tea']]`
- `strs = ['']` → `[['']]`
- `strs = ['a']` → `[['a']]`

In [ ]:
def group_anagrams(strs):
    """
    Group the strings in strs so that anagrams are together. The groups may be
    returned in any order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(group_anagrams, [((["eat", "tea", "tan", "ate", "nat", "bat"],), [["bat"], ["nat", "tan"], ["ate", "eat", "tea"]]), (([""],), [[""]]), ((["a"],), [["a"]])], norm=lambda g: sorted(sorted(x) for x in g))

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each word not yet grouped, compare it with every other word by sorting both.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Give every word a key that is identical for anagrams — the tuple of its 26 letter counts — and bucket words by key in a dict.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each word not yet grouped, compare it with every other word by sorting both.

```python
def group_anagrams(strs):
    used = [False] * len(strs)
    groups = []
    for i in range(len(strs)):
        if used[i]:
            continue
        group = [strs[i]]
        used[i] = True
        for j in range(i + 1, len(strs)):
            if not used[j] and sorted(strs[i]) == sorted(strs[j]):
                group.append(strs[j])
                used[j] = True
        groups.append(group)
    return groups
```

**Cost:** O(n² · k log k) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Give every word a key that is identical for anagrams — the tuple of its 26 letter counts — and bucket words by key in a dict.

```python
def group_anagrams(strs):
    groups = {}
    for i, w in enumerate(strs):
        count = [0] * 26
        for ch in w:
            count[ord(ch) - ord("a")] += 1
        key = tuple(count)
        groups.setdefault(key, []).append(w)
    return list(groups.values())
```

**Cost:** O(n · k) time, O(n · k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each word not yet grouped, compare it with every other word by sorting both. That costs O(n² · k log k) time and O(n) space."

"I can do better: Give every word a key that is identical for anagrams — the tuple of its 26 letter counts — and bucket words by key in a dict. That's O(n · k) time and O(n · k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `strs = ['eat', 'tea', 'tan', 'nat', 'bat']`

1. 'eat' → key aet (its letter counts)
2. 'tea' → key aet (its letter counts)
3. 'tan' → key ant (its letter counts)
4. 'nat' → key ant (its letter counts)
5. 'bat' → key abt (its letter counts)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/group-anagrams)

<a id="q9"></a>
### Q9. Top K Frequent Elements

🟡 Interview standard · *Bucket sort*

Return the `k` most frequent elements of `nums`, in any order.

**Examples**

- `nums = [1, 1, 1, 2, 2, 3], k = 2` → `[1, 2]`
- `nums = [1], k = 1` → `[1]`
- `nums = [4, 4, 5, 5, 5, 6], k = 1` → `[5]`

In [ ]:
def top_k_frequent(nums, k):
    """
    Return the k most frequent elements of nums, in any order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(top_k_frequent, [(([1, 1, 1, 2, 2, 3], 2), [1, 2]), (([1], 1), [1]), (([4, 4, 5, 5, 5, 6], 1), [5])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count frequencies, sort the distinct values by count, take the first k.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A count is between 1 and n, so make n + 1 buckets where bucket[c] lists values seen c times. Walk buckets from high to low until k values are collected.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Count frequencies, sort the distinct values by count, take the first k.

```python
def top_k_frequent(nums, k):
    count = {}
    for x in nums:
        count[x] = count.get(x, 0) + 1
    ranked = sorted(count, key=lambda x: -count[x])
    return ranked[:k]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** A count is between 1 and n, so make n + 1 buckets where bucket[c] lists values seen c times. Walk buckets from high to low until k values are collected.

```python
def top_k_frequent(nums, k):
    count = {}
    for x in nums:
        count[x] = count.get(x, 0) + 1
    buckets = [[] for _ in range(len(nums) + 1)]
    for x, c in count.items():
        buckets[c].append(x)
    res = []
    for c in range(len(buckets) - 1, 0, -1):
        for x in buckets[c]:
            res.append(x)
            if len(res) == k:
                return res
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count frequencies, sort the distinct values by count, take the first k. That costs O(n log n) time and O(n) space."

"I can do better: A count is between 1 and n, so make n + 1 buckets where bucket[c] lists values seen c times. Walk buckets from high to low until k values are collected. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 1, 1, 2, 2, 3], k = 2`

1. Bucket c holds the values that appear c times
2. Take 1 (appears 3 times)
3. Take 2 (appears 2 times)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/top-k-frequent-elements)

<a id="q10"></a>
### Q10. Product of Array Except Self

🟡 Interview standard · *Prefix / suffix*

Return `answer` where `answer[i]` is the product of every element of `nums` except `nums[i]`, without using division.

**Examples**

- `nums = [1, 2, 3, 4]` → `[24, 12, 8, 6]`
- `nums = [-1, 1, 0, -3, 3]` → `[0, 0, 9, 0, 0]`

In [ ]:
def product_except_self(nums):
    """
    Return answer where answer[i] is the product of every element of nums except
    nums[i], without using division.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(product_except_self, [
    (([1, 2, 3, 4],), [24, 12, 8, 6]),
    (([-1, 1, 0, -3, 3],), [0, 0, 9, 0, 0]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each i, multiply every other element.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

answer[i] = (product of everything left of i) × (product of everything right of i). Fill the left products in one pass, then sweep from the right with a running product.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each i, multiply every other element.

```python
def product_except_self(nums):
    n = len(nums)
    answer = [1] * n
    for i in range(n):
        for j in range(n):
            if i != j:
                answer[i] *= nums[j]
    return answer
```

**Cost:** O(n²) time, O(1) extra space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** answer[i] = (product of everything left of i) × (product of everything right of i). Fill the left products in one pass, then sweep from the right with a running product.

```python
def product_except_self(nums):
    n = len(nums)
    answer = [1] * n
    left = 1
    for i in range(n):
        answer[i] = left
        left *= nums[i]
    right = 1
    for i in range(n - 1, -1, -1):
        answer[i] *= right
        right *= nums[i]
    return answer
```

**Cost:** O(n) time, O(1) extra (output not counted) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each i, multiply every other element. That costs O(n²) time and O(1) extra space."

"I can do better: answer[i] = (product of everything left of i) × (product of everything right of i). Fill the left products in one pass, then sweep from the right with a running product. That's O(n) time and O(1) extra (output not counted) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 2, 3, 4]`

1. answer[0] = product left of 0 = 1
2. answer[1] = product left of 1 = 1
3. answer[2] = product left of 2 = 2
4. answer[3] = product left of 3 = 6
5. answer[3] ×= product right of 3 = 1
6. answer[2] ×= product right of 2 = 4
7. answer[1] ×= product right of 1 = 12
8. answer[0] ×= product right of 0 = 24

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/product-of-array-except-self)

<a id="q11"></a>
### Q11. Longest Consecutive Sequence

🟡 Interview standard · *Hash set*

Given an unsorted array `nums`, return the length of the longest run of consecutive integers (like 1, 2, 3, 4). Must run in O(n).

**Examples**

- `nums = [100, 4, 200, 1, 3, 2]` → `4`
- `nums = [0, 3, 7, 2, 5, 8, 4, 6, 0, 1]` → `9`
- `nums = []` → `0`

In [ ]:
def longest_consecutive(nums):
    """
    Given an unsorted array nums, return the length of the longest run of
    consecutive integers (like 1, 2, 3, 4). Must run in O(n).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(longest_consecutive, [
    (([100, 4, 200, 1, 3, 2],), 4),
    (([0, 3, 7, 2, 5, 8, 4, 6, 0, 1],), 9),
    (([],), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For every x, keep checking whether x + 1, x + 2, … exist by scanning the list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Put everything in a set. Only start counting at numbers that begin a run (x − 1 not in the set), then walk x + 1, x + 2, … in the set. Each number is visited at most twice.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For every x, keep checking whether x + 1, x + 2, … exist by scanning the list.

```python
def longest_consecutive(nums):
    best = 0
    for x in nums:
        length = 1
        while x + length in nums:
            length += 1
        best = max(best, length)
    return best
```

**Cost:** O(n³) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Put everything in a set. Only start counting at numbers that begin a run (x − 1 not in the set), then walk x + 1, x + 2, … in the set. Each number is visited at most twice.

```python
def longest_consecutive(nums):
    s = set(nums)
    best = 0
    for x in s:
        if x - 1 in s:
            continue
        length = 1
        while x + length in s:
            length += 1
        best = max(best, length)
    return best
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For every x, keep checking whether x + 1, x + 2, … exist by scanning the list. That costs O(n³) time and O(1) space."

"I can do better: Put everything in a set. Only start counting at numbers that begin a run (x − 1 not in the set), then walk x + 1, x + 2, … in the set. Each number is visited at most twice. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [100, 4, 200, 1, 3, 2]`

1. 1 starts a run: 1..4, length 4
2. 1 exists, so 2 is not the start of a run — skip
3. 2 exists, so 3 is not the start of a run — skip
4. 100 starts a run: 100..100, length 1
5. 3 exists, so 4 is not the start of a run — skip
6. 200 starts a run: 200..200, length 1

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/longest-consecutive-sequence)

<a id="q12"></a>
### Q12. Maximum Subarray

🟡 Interview standard · *Kadane*

Find the contiguous subarray with the largest sum and return that sum.

**Examples**

- `nums = [-2, 1, -3, 4, -1, 2, 1, -5, 4]` → `6`
- `nums = [1]` → `1`
- `nums = [5, 4, -1, 7, 8]` → `23`

In [ ]:
def max_subarray(nums):
    """
    Find the contiguous subarray with the largest sum and return that sum.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_subarray, [
    (([-2, 1, -3, 4, -1, 2, 1, -5, 4],), 6),
    (([1],), 1),
    (([5, 4, -1, 7, 8],), 23),
    (([-3, -1, -2],), -1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Fix each start i, extend the end j and keep a running sum.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Kadane: cur = best sum of a subarray ending at i. Either extend the previous one or start fresh at nums[i]: cur = max(nums[i], cur + nums[i]).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Fix each start i, extend the end j and keep a running sum.

```python
def max_subarray(nums):
    best = nums[0]
    for i in range(len(nums)):
        total = 0
        for j in range(i, len(nums)):
            total += nums[j]
            best = max(best, total)
    return best
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Kadane: cur = best sum of a subarray ending at i. Either extend the previous one or start fresh at nums[i]: cur = max(nums[i], cur + nums[i]).

```python
def max_subarray(nums):
    cur = best = nums[0]
    start = bs = be = 0
    for i in range(1, len(nums)):
        if cur + nums[i] < nums[i]:
            cur, start = nums[i], i
        else:
            cur += nums[i]
        if cur > best:
            best, bs, be = cur, start, i
    return best
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Fix each start i, extend the end j and keep a running sum. That costs O(n²) time and O(1) space."

"I can do better: Kadane: cur = best sum of a subarray ending at i. Either extend the previous one or start fresh at nums[i]: cur = max(nums[i], cur + nums[i]). That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [-2, 1, -3, 4, -1, 2, 1, -5, 4]`

1. Best sum ending at 1 is 1; best overall 1
2. Best sum ending at 2 is -2; best overall 1
3. Best sum ending at 3 is 4; best overall 4
4. Best sum ending at 4 is 3; best overall 4
5. Best sum ending at 5 is 5; best overall 5
6. Best sum ending at 6 is 6; best overall 6
7. Best sum ending at 7 is 1; best overall 6
8. Best sum ending at 8 is 5; best overall 6

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/maximum-subarray)

<a id="q13"></a>
### Q13. Rotate Array

🟡 Interview standard · *Reverse trick*

Rotate `nums` to the right by `k` steps, in place.

**Examples**

- `nums = [1, 2, 3, 4, 5, 6, 7], k = 3` → `[5, 6, 7, 1, 2, 3, 4]`
- `nums = [-1, -100, 3, 99], k = 2` → `[3, 99, -1, -100]`
- `nums = [1, 2], k = 5` → `[2, 1]`

In [ ]:
def rotate(nums, k):
    """
    Rotate nums to the right by k steps, in place.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rotate, [(([1, 2, 3, 4, 5, 6, 7], 3), [5, 6, 7, 1, 2, 3, 4]), (([-1, -100, 3, 99], 2), [3, 99, -1, -100]), (([1, 2], 5), [2, 1])], inplace=0)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Rotate by one step, k times.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Reverse the whole array, then reverse the first k and the last n − k elements. The last k elements end up in front, each block in its original order.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Rotate by one step, k times.

```python
def rotate(nums, k):
    for _ in range(k % len(nums)):
        nums.insert(0, nums.pop())
```

**Cost:** O(n · k) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Reverse the whole array, then reverse the first k and the last n − k elements. The last k elements end up in front, each block in its original order.

```python
def rotate(nums, k):
    n = len(nums)
    k %= n

    def rev(l, r):
        while l < r:
            nums[l], nums[r] = nums[r], nums[l]
            l, r = l + 1, r - 1

    rev(0, n - 1)
    rev(0, k - 1)
    rev(k, n - 1)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Rotate by one step, k times. That costs O(n · k) time and O(1) space."

"I can do better: Reverse the whole array, then reverse the first k and the last n − k elements. The last k elements end up in front, each block in its original order. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 2, 3, 4, 5, 6, 7], k = 3`

1. 1) Reverse everything
2. 2) Reverse the first k = 3
3. 3) Reverse the rest — done

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/rotate-array)

<a id="q14"></a>
### Q14. Subarray Sum Equals K

🟡 Interview standard · *Prefix sum + hash map*

Return the number of contiguous subarrays whose sum equals `k`. Values may be negative.

**Examples**

- `nums = [1, 1, 1], k = 2` → `2`
- `nums = [1, 2, 3], k = 3` → `2`
- `nums = [1, -1, 0], k = 0` → `3`

In [ ]:
def subarray_sum(nums, k):
    """
    Return the number of contiguous subarrays whose sum equals k. Values may be
    negative.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(subarray_sum, [
    (([1, 1, 1], 2), 2),
    (([1, 2, 3], 3), 2),
    (([1, -1, 0], 0), 3),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Fix each start, extend the end, count the sums equal to k.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

With running sum pre, a subarray ending here sums to k when an earlier prefix equals pre − k. Keep a count of every prefix sum seen; start with {0: 1}.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Fix each start, extend the end, count the sums equal to k.

```python
def subarray_sum(nums, k):
    count = 0
    for i in range(len(nums)):
        total = 0
        for j in range(i, len(nums)):
            total += nums[j]
            if total == k:
                count += 1
    return count
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** With running sum pre, a subarray ending here sums to k when an earlier prefix equals pre − k. Keep a count of every prefix sum seen; start with {0: 1}.

```python
def subarray_sum(nums, k):
    seen = {0: 1}  # prefix sum -> how many times
    pre = count = 0
    for i, x in enumerate(nums):
        pre += x
        count += seen.get(pre - k, 0)
        seen[pre] = seen.get(pre, 0) + 1
    return count
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Fix each start, extend the end, count the sums equal to k. That costs O(n²) time and O(1) space."

"I can do better: With running sum pre, a subarray ending here sums to k when an earlier prefix equals pre − k. Keep a count of every prefix sum seen; start with {0: 1}. That's O(n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [3, 4, -7, 1, 3, 3, 1, -4], k = 7`

1. prefix = 3. Earlier prefixes equal to 3 − 7 = -4: 0
2. prefix = 7. Earlier prefixes equal to 7 − 7 = 0: 1
3. prefix = 0. Earlier prefixes equal to 0 − 7 = -7: 0
4. prefix = 1. Earlier prefixes equal to 1 − 7 = -6: 0
5. prefix = 4. Earlier prefixes equal to 4 − 7 = -3: 0
6. prefix = 7. Earlier prefixes equal to 7 − 7 = 0: 2
7. prefix = 8. Earlier prefixes equal to 8 − 7 = 1: 1
8. prefix = 4. Earlier prefixes equal to 4 − 7 = -3: 0

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/subarray-sum-equals-k)

<a id="q15"></a>
### Q15. Sort Colors

🟡 Interview standard · *Dutch national flag*

Sort an array of 0s, 1s and 2s in place in one pass, without the library sort.

**Examples**

- `nums = [2, 0, 2, 1, 1, 0]` → `[0, 0, 1, 1, 2, 2]`
- `nums = [2, 0, 1]` → `[0, 1, 2]`

In [ ]:
def sort_colors(nums):
    """
    Sort an array of 0s, 1s and 2s in place in one pass, without the library
    sort.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sort_colors, [(([2, 0, 2, 1, 1, 0],), [0, 0, 1, 1, 2, 2]), (([2, 0, 1],), [0, 1, 2])], inplace=0)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count the 0s, 1s and 2s, then overwrite the array.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Three pointers: everything before lo is 0, everything after hi is 2, mid scans. 0 → swap to lo; 2 → swap to hi (don't advance mid); 1 → advance.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Count the 0s, 1s and 2s, then overwrite the array.

```python
def sort_colors(nums):
    c = [nums.count(0), nums.count(1), nums.count(2)]
    nums[:] = [0] * c[0] + [1] * c[1] + [2] * c[2]
```

**Cost:** O(n) — two passes time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Three pointers: everything before lo is 0, everything after hi is 2, mid scans. 0 → swap to lo; 2 → swap to hi (don't advance mid); 1 → advance.

```python
def sort_colors(nums):
    lo, mid, hi = 0, 0, len(nums) - 1
    while mid <= hi:
        if nums[mid] == 0:
            nums[lo], nums[mid] = nums[mid], nums[lo]
            lo, mid = lo + 1, mid + 1
        elif nums[mid] == 2:
            nums[mid], nums[hi] = nums[hi], nums[mid]
            hi -= 1
        else:
            mid += 1
```

**Cost:** O(n) — one pass time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count the 0s, 1s and 2s, then overwrite the array. That costs O(n) — two passes time and O(1) space."

"I can do better: Three pointers: everything before lo is 0, everything after hi is 2, mid scans. 0 → swap to lo; 2 → swap to hi (don't advance mid); 1 → advance. That's O(n) — one pass time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [2, 0, 2, 1, 1, 0]`

1. nums[mid] = 2
2. nums[mid] = 0
3. nums[mid] = 0
4. nums[mid] = 2
5. nums[mid] = 1
6. nums[mid] = 1
7. Sorted

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/sort-colors)

<a id="q16"></a>
### Q16. Next Permutation

🟡 Interview standard · *Pivot + reverse*

Rearrange `nums` in place into the next lexicographically greater permutation. If it is the largest, wrap to the smallest (sorted).

**Examples**

- `nums = [1, 2, 3]` → `[1, 3, 2]`
- `nums = [3, 2, 1]` → `[1, 2, 3]`
- `nums = [1, 1, 5]` → `[1, 5, 1]`

In [ ]:
def next_permutation(nums):
    """
    Rearrange nums in place into the next lexicographically greater permutation.
    If it is the largest, wrap to the smallest (sorted).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(next_permutation, [(([1, 2, 3],), [1, 3, 2]), (([3, 2, 1],), [1, 2, 3]), (([1, 1, 5],), [1, 5, 1]), (([1, 3, 2],), [2, 1, 3])], inplace=0)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Generate every permutation in sorted order and pick the one after the current.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

From the right, find the first i with nums[i] &lt; nums[i+1] (the pivot). Swap it with the smallest larger element to its right, then reverse the suffix after i.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Generate every permutation in sorted order and pick the one after the current.

```python
def next_permutation(nums):
    from itertools import permutations
    perms = sorted(set(permutations(nums)))
    i = perms.index(tuple(nums))
    nums[:] = perms[(i + 1) % len(perms)]
```

**Cost:** O(n! · n) time, O(n! · n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** From the right, find the first i with nums[i] &lt; nums[i+1] (the pivot). Swap it with the smallest larger element to its right, then reverse the suffix after i.

```python
def next_permutation(nums):
    i = len(nums) - 2
    while i >= 0 and nums[i] >= nums[i + 1]:
        i -= 1
    if i >= 0:
        j = len(nums) - 1
        while nums[j] <= nums[i]:
            j -= 1
        nums[i], nums[j] = nums[j], nums[i]
    nums[i + 1:] = reversed(nums[i + 1:])
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Generate every permutation in sorted order and pick the one after the current. That costs O(n! · n) time and O(n! · n) space."

"I can do better: From the right, find the first i with nums[i] &lt; nums[i+1] (the pivot). Swap it with the smallest larger element to its right, then reverse the suffix after i. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 3, 5, 4, 2]`

1. Pivot: first index from the right with nums[i] &lt; nums[i+1] → i = 1
2. Swap pivot with the smallest larger value on its right (index 3)
3. Reverse the suffix so it is as small as possible

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/next-permutation)

<a id="q17"></a>
### Q17. Find the Duplicate Number

🟡 Interview standard · *Floyd cycle*

`nums` has n + 1 integers, each in 1..n, and exactly one value repeats. Find it without modifying the array, using O(1) extra space.

**Examples**

- `nums = [1, 3, 4, 2, 2]` → `2`
- `nums = [3, 1, 3, 4, 2]` → `3`
- `nums = [3, 3, 3, 3, 3]` → `3`

In [ ]:
def find_duplicate(nums):
    """
    nums has n + 1 integers, each in 1..n, and exactly one value repeats. Find
    it without modifying the array, using O(1) extra space.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(find_duplicate, [
    (([1, 3, 4, 2, 2],), 2),
    (([3, 1, 3, 4, 2],), 3),
    (([3, 3, 3, 3, 3],), 3),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Remember values in a set and return the first repeat.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Treat i → nums[i] as a linked list. Two indices pointing at the same value make a cycle whose entrance is the duplicate. Find it with Floyd's tortoise and hare.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Remember values in a set and return the first repeat.

```python
def find_duplicate(nums):
    seen = set()
    for x in nums:
        if x in seen:
            return x
        seen.add(x)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Treat i → nums[i] as a linked list. Two indices pointing at the same value make a cycle whose entrance is the duplicate. Find it with Floyd's tortoise and hare.

```python
def find_duplicate(nums):
    slow = fast = nums[0]
    while True:
        slow, fast = nums[slow], nums[nums[fast]]
        if slow == fast:
            break
    slow = nums[0]
    while slow != fast:
        slow, fast = nums[slow], nums[fast]
    return slow
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Remember values in a set and return the first repeat. That costs O(n) time and O(n) space."

"I can do better: Treat i → nums[i] as a linked list. Two indices pointing at the same value make a cycle whose entrance is the duplicate. Find it with Floyd's tortoise and hare. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 3, 4, 2, 2]`

1. Phase 1: slow moves 1 step, fast 2 steps
2. Phase 1: slow moves 1 step, fast 2 steps
3. Phase 2: both move 1 step until they meet at the cycle entrance
4. Phase 2: both move 1 step until they meet at the cycle entrance
5. They meet at index 2 → the duplicate value is 2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/find-the-duplicate-number)

<a id="q18"></a>
### Q18. Set Matrix Zeroes

🟡 Interview standard · *In-place markers*

If a cell of an m × n matrix is 0, set its whole row and column to 0. Do it in place.

**Examples**

- `matrix = [[1, 1, 1], [1, 0, 1], [1, 1, 1]]` → `[[1, 0, 1], [0, 0, 0], [1, 0, 1]]`
- `matrix = [[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]` → `[[0, 0, 0, 0], [0, 4, 5, 0], [0, 3, 1, 0]]`

In [ ]:
def set_zeroes(matrix):
    """
    If a cell of an m × n matrix is 0, set its whole row and column to 0. Do it
    in place.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(set_zeroes, [(([[1, 1, 1], [1, 0, 1], [1, 1, 1]],), [[1, 0, 1], [0, 0, 0], [1, 0, 1]]),
   (([[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]],), [[0, 0, 0, 0], [0, 4, 5, 0], [0, 3, 1, 0]])], inplace=0)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Record which rows and columns contain a zero in two sets, then zero them out.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Use the first row and first column as the marker arrays. Remember separately whether row 0 itself had a zero, mark, zero the inner cells, then fix row 0 and column 0 last.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Record which rows and columns contain a zero in two sets, then zero them out.

```python
def set_zeroes(matrix):
    rows, cols = set(), set()
    for r, row in enumerate(matrix):
        for c, v in enumerate(row):
            if v == 0:
                rows.add(r)
                cols.add(c)
    for r in range(len(matrix)):
        for c in range(len(matrix[0])):
            if r in rows or c in cols:
                matrix[r][c] = 0
```

**Cost:** O(m · n) time, O(m + n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Use the first row and first column as the marker arrays. Remember separately whether row 0 itself had a zero, mark, zero the inner cells, then fix row 0 and column 0 last.

```python
def set_zeroes(matrix):
    m, n = len(matrix), len(matrix[0])
    row0 = any(matrix[0][c] == 0 for c in range(n))
    for r in range(1, m):
        for c in range(n):
            if matrix[r][c] == 0:
                matrix[r][0] = matrix[0][c] = 0
    for r in range(1, m):
        for c in range(n - 1, -1, -1):
            if matrix[r][0] == 0 or matrix[0][c] == 0:
                matrix[r][c] = 0
    if row0:
        for c in range(n):
            matrix[0][c] = 0
```

**Cost:** O(m · n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Record which rows and columns contain a zero in two sets, then zero them out. That costs O(m · n) time and O(m + n) space."

"I can do better: Use the first row and first column as the marker arrays. Remember separately whether row 0 itself had a zero, mark, zero the inner cells, then fix row 0 and column 0 last. That's O(m · n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `matrix = [[1, 1, 1], [1, 0, 1], [1, 1, 1]]`

1. Zero at (1,1) → mark row 1 in column 0 and column 1 in row 0
2. Zero every cell whose row or column is marked
3. Finally fix row 0 if it had a zero of its own

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/set-matrix-zeroes)

<a id="q19"></a>
### Q19. Spiral Matrix

🟡 Interview standard · *Boundary shrink*

Return all elements of an m × n matrix in spiral order (right, down, left, up, repeat).

**Examples**

- `matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]` → `[1, 2, 3, 6, 9, 8, 7, 4, 5]`
- `matrix = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]` → `[1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]`

In [ ]:
def spiral_order(matrix):
    """
    Return all elements of an m × n matrix in spiral order (right, down, left,
    up, repeat).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(spiral_order, [
    (([[1, 2, 3], [4, 5, 6], [7, 8, 9]],), [1, 2, 3, 6, 9, 8, 7, 4, 5]),
    (([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]],), [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk the grid with a direction and a visited matrix, turning right when blocked.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep four walls top, bottom, left, right. Read the top row, right column, bottom row, left column, shrinking the matching wall after each.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Walk the grid with a direction and a visited matrix, turning right when blocked.

```python
def spiral_order(matrix):
    m, n = len(matrix), len(matrix[0])
    seen = [[False] * n for _ in range(m)]
    dr, dc = [0, 1, 0, -1], [1, 0, -1, 0]
    r = c = d = 0
    out = []
    for _ in range(m * n):
        out.append(matrix[r][c])
        seen[r][c] = True
        nr, nc = r + dr[d], c + dc[d]
        if not (0 <= nr < m and 0 <= nc < n) or seen[nr][nc]:
            d = (d + 1) % 4
            nr, nc = r + dr[d], c + dc[d]
        r, c = nr, nc
    return out
```

**Cost:** O(m · n) time, O(m · n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Keep four walls top, bottom, left, right. Read the top row, right column, bottom row, left column, shrinking the matching wall after each.

```python
def spiral_order(matrix):
    out = []
    top, bottom, left, right = 0, len(matrix) - 1, 0, len(matrix[0]) - 1
    done = []
    while top <= bottom and left <= right:
        for c in range(left, right + 1):
            out.append(matrix[top][c]); done.append((top, c))
        top += 1
        for r in range(top, bottom + 1):
            out.append(matrix[r][right]); done.append((r, right))
        right -= 1
        if top <= bottom:
            for c in range(right, left - 1, -1):
                out.append(matrix[bottom][c]); done.append((bottom, c))
            bottom -= 1
        if left <= right:
            for r in range(bottom, top - 1, -1):
                out.append(matrix[r][left]); done.append((r, left))
            left += 1
    return out
```

**Cost:** O(m · n) time, O(1) extra space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk the grid with a direction and a visited matrix, turning right when blocked. That costs O(m · n) time and O(m · n) space."

"I can do better: Keep four walls top, bottom, left, right. Read the top row, right column, bottom row, left column, shrinking the matching wall after each. That's O(m · n) time and O(1) extra space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]`

1. Top row, left → right
2. Right column, top → bottom
3. Bottom row, right → left
4. Left column, bottom → top
5. Top row, left → right
6. Right column, top → bottom

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/spiral-matrix)

<a id="q20"></a>
### Q20. Rotate Image

🟡 Interview standard · *Transpose + reverse*

Rotate an n × n matrix 90° clockwise, in place.

**Examples**

- `matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]` → `[[7, 4, 1], [8, 5, 2], [9, 6, 3]]`
- `matrix = [[5, 1, 9, 11], [2, 4, 8, 10], [13, 3, 6, 7], [15, 14, 12, 16]]` → `[[15, 13, 2, 5], [14, 3, 4, 1], [12, 6, 8, 9], [16, 7, 10, 11]]`

In [ ]:
def rotate_image(matrix):
    """
    Rotate an n × n matrix 90° clockwise, in place.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rotate_image, [(([[1, 2, 3], [4, 5, 6], [7, 8, 9]],), [[7, 4, 1], [8, 5, 2], [9, 6, 3]]),
   (([[5, 1, 9, 11], [2, 4, 8, 10], [13, 3, 6, 7], [15, 14, 12, 16]],), [[15, 13, 2, 5], [14, 3, 4, 1], [12, 6, 8, 9], [16, 7, 10, 11]])], inplace=0)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build a new matrix where new[c][n−1−r] = old[r][c], then copy it back.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A clockwise turn equals a transpose (swap across the main diagonal) followed by reversing each row.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Build a new matrix where new[c][n−1−r] = old[r][c], then copy it back.

```python
def rotate_image(matrix):
    n = len(matrix)
    new = [[0] * n for _ in range(n)]
    for r in range(n):
        for c in range(n):
            new[c][n - 1 - r] = matrix[r][c]
    matrix[:] = new
```

**Cost:** O(n²) time, O(n²) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** A clockwise turn equals a transpose (swap across the main diagonal) followed by reversing each row.

```python
def rotate_image(matrix):
    n = len(matrix)
    for r in range(n):
        for c in range(r + 1, n):
            matrix[r][c], matrix[c][r] = matrix[c][r], matrix[r][c]
    for row in matrix:
        row.reverse()
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build a new matrix where new[c][n−1−r] = old[r][c], then copy it back. That costs O(n²) time and O(n²) space."

"I can do better: A clockwise turn equals a transpose (swap across the main diagonal) followed by reversing each row. That's O(n²) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]`

1. Transpose: swap (0,1) ↔ (1,0)
2. Transpose: swap (0,2) ↔ (2,0)
3. Transpose: swap (1,2) ↔ (2,1)
4. Reverse every row → rotated 90° clockwise

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/arrays/rotate-image)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Why does array indexing start at 0?</b></summary>

The index is an **offset** from the start address: element i lives at `base + i × size`. The first element is at offset 0, so its index is 0. This avoids an extra subtraction on every access.

</details>

<details>
<summary><b>2. What is the difference between a static and a dynamic array?</b></summary>

A static array has a fixed size chosen at creation. A dynamic array (Python list) grows automatically: when full, it allocates a bigger block (about 1.125–2× in practice), copies the elements, and frees the old block. Appends are O(1) amortised.

</details>

<details>
<summary><b>3. Explain amortised O(1) for append.</b></summary>

Most appends just write into spare capacity (O(1)). Occasionally the array is full and must copy all n elements (O(n)). Because the capacity doubles, those expensive copies happen rarely: n appends cost under 2n total work, so the **average** per append is O(1).

</details>

<details>
<summary><b>4. How does a hash map achieve O(1) lookup? When is it O(n)?</b></summary>

The hash function turns the key into a bucket index, so lookup checks one bucket instead of every element. With a good hash function and a bounded load factor, buckets hold O(1) entries on average. If many keys collide into the same bucket (bad hash or an attack), lookup degrades to O(n).

</details>

<details>
<summary><b>5. What is a collision and how is it handled?</b></summary>

Two keys hashing to the same bucket. **Chaining** stores a list in each bucket; **open addressing** probes for another free slot (linear, quadratic, double hashing). Both keep working correctly; they differ in memory use and cache behaviour.

</details>

<details>
<summary><b>6. Why can't a Python list be a dictionary key?</b></summary>

Lists are mutable. If a list changed after insertion, its hash would change and the dict would look in the wrong bucket. Only immutable (hashable) types — int, str, tuple of hashables, frozenset — can be keys. Convert with `tuple(lst)`.

</details>

<details>
<summary><b>7. Subarray vs subsequence vs subset?</b></summary>

**Subarray**: contiguous piece, order kept (n(n+1)/2 of them). **Subsequence**: any elements, order kept, gaps allowed (2ⁿ). **Subset**: any elements, order doesn't matter (2ⁿ).

</details>

<details>
<summary><b>8. What is the time complexity of x in list vs x in set?</b></summary>

`x in list` scans the list: O(n). `x in set` hashes x and checks one bucket: O(1) on average. Inside a loop this is the difference between O(n²) and O(n).

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: What an array really is
- [ ] I can explain: Insert, delete and search — what they cost
- [ ] I can explain: Hash maps and hash sets
- [ ] I can explain: Pattern: remember what you have seen
- [ ] I can explain: Prefix sums
- [ ] I can explain: Running state: Kadane and one-pass scans
- [ ] I can explain: In-place tricks and matrices

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Next Permutation
- [ ] [Q17](#q17) Find the Duplicate Number
- [ ] [Q18](#q18) Set Matrix Zeroes
- [ ] [Q19](#q19) Spiral Matrix
- [ ] [Q20](#q20) Rotate Image

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.